In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Conv2D,MaxPooling2D,Flatten,Dropout,BatchNormalization,GlobalAveragePooling2D
from keras.applications import ResNet50
from keras.models import Model

from keras.layers import RandomFlip, RandomRotation, RandomZoom,RandomTranslation,RandomContrast, Input
from keras.callbacks import EarlyStopping,ModelCheckpoint

import sklearn
from sklearn.model_selection import train_test_split

import cv2


In [3]:
df = pd.read_csv('./data/fer2013.csv')

In [4]:
df.head()

,emotion,pixels,Usage
0,0,70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...,Training
1,0,151 150 147 155 148 133 111 140 170 174 182 15...,Training
2,2,231 212 156 164 174 138 161 173 182 200 106 38...,Training
3,4,24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...,Training
4,6,4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...,Training


In [5]:
#0: Angry
#1: Disgust
#2: Fear
#3: Happy
#4: Sad
#5: Surprise
#6: Neutral

In [6]:
df['emotion'].value_counts()

emotion
3    8989
6    6198
4    6077
2    5121
0    4953
5    4002
1     547
Name: count, dtype: int64

In [9]:
train_data = df[df['Usage'] == 'Training']
val_data = df[df['Usage'] == 'PublicTest']
test_data = df[df['Usage'] == 'PrivateTest']


def pixel_preprocess(df):


    x=[]
    y=[]

    for index,row in df.iterrows():  # iterating over all rows
        pixels = np.array(row['pixels'].split(),dtype='float32')
        pixels = pixels.reshape(48,48,1)
        x.append(pixels)
        y.append(row['emotion'])

    x = np.array(x)  
    y = keras.utils.to_categorical(np.array(y), num_classes=7)
    return x,y

X_train,y_train = pixel_preprocess(train_data)
X_val,y_val = pixel_preprocess(val_data)
X_test,y_test = pixel_preprocess(test_data)


print("Train",X_train.shape,y_train.shape)
print("Val",X_val.shape,y_val.shape)
print("Test",X_test.shape,y_test.shape)


Train (28709, 48, 48, 1) (28709, 7)
Val (3589, 48, 48, 1) (3589, 7)
Test (3589, 48, 48, 1) (3589, 7)


In [11]:
#tried to add transfer learning but rehsape garda ali dherai spcae khayo
# so paxi  kapur ko laptop ma sangai huda garne 

In [ ]:
# dont run this take up lot of fucking space (8gb)

 def resize(images):
    images = tf.convert_to_tensor(images, dtype=tf.float32)
    images = tf.image.grayscale_to_rgb(images)     # (48,48,1) -> (48,48,3)
    images = tf.image.resize(images, (224,224))    # (48,48,3) -> (224,224,3)  cause Resnet expects that image
    images = images / 255.0
    return images

X_train = resize(X_train)
X_test = resize(X_test)
X_val = resize(X_val)

MemoryError: Unable to allocate 252. MiB for an array with shape (28709, 48, 48, 1) and data type float32

In [9]:
X_train = X_train / 255.0
X_val   = X_val / 255.0
X_test  = X_test / 255.0

In [ ]:
data_aug = tf.keras.Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.06),
    RandomZoom(0.08),
    RandomTranslation(0.06, 0.06),
    RandomContrast(0,15)
])

In [14]:
base_model = tf.keras.applications.ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

base_model.trainable = False

In [17]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128,activation="relu")(x)
x = Dropout(0.5)(x)

outputs = Dense(7,activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=outputs)

model=Sequential()

model.add(Input(shape=(48,48,1)))
model.add(data_aug)

model.add(Conv2D(32,kernel_size=(3,3),padding='same',activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(32,kernel_size=(3,3),padding='same',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding="same"))
model.add(Dropout(0.25))

model.add(Conv2D(64,kernel_size=(3,3),padding='same',activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(64,kernel_size=(3,3),padding='same',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='same'))
model.add(Dropout(0.25))

model.add(Conv2D(64,kernel_size=(3,3),padding='same',activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(64,kernel_size=(3,3),padding='same',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='same'))
model.add(Dropout(0.3))


model.add(GlobalAveragePooling2D())

model.add(Dense(128,activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(7,activation='softmax'))


In [11]:
early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

checkpoint = ModelCheckpoint('emotion_model.h5',monitor='val_loss',save_best_only=True)

In [18]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,850,887 (90.98 MB)

 Trainable params: 263,175 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [19]:
loss = keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

In [ ]:
optimizer = keras.optimizers.Adam(learning_rate=1e-3)

model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=['accuracy']
)

In [14]:
history = model.fit(
    X_train,y_train,
    epochs=60,
    batch_size=64,
    shuffle=True,
    validation_data=(X_val,y_val),
    callbacks=[early_stop,checkpoint]
    )


Epoch 1/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 0.2124 - loss: 1.9257

449/449 ━━━━━━━━━━━━━━━━━━━━ 65s 139ms/step - accuracy: 0.2307 - loss: 1.8529 - val_accuracy: 0.2494 - val_loss: 1.8132
Epoch 2/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 0.2545 - loss: 1.7828

449/449 ━━━━━━━━━━━━━━━━━━━━ 62s 138ms/step - accuracy: 0.2613 - loss: 1.7772 - val_accuracy: 0.2906 - val_loss: 1.7216
Epoch 3/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 0.2907 - loss: 1.7369

449/449 ━━━━━━━━━━━━━━━━━━━━ 62s 138ms/step - accuracy: 0.3003 - loss: 1.7227 - val_accuracy: 0.3505 - val_loss: 1.6226
Epoch 4/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 0.3351 - loss: 1.6741

449/449 ━━━━━━━━━━━━━━━━━━━━ 62s 138ms/step - accuracy: 0.3391 - loss: 1.6645 - val_accuracy: 0.4246 - val_loss: 1.5153
Epoch 5/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - accuracy: 0.3551 - loss: 1.6279

...

449/449 ━━━━━━━━━━━━━━━━━━━━ 61s 136ms/step - accuracy: 0.4951 - loss: 1.3184 - val_accuracy: 0.5553 - val_loss: 1.1738
Epoch 26/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 61s 136ms/step - accuracy: 0.5024 - loss: 1.3180 - val_accuracy: 0.5355 - val_loss: 1.2230
Epoch 27/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.4976 - loss: 1.3161

449/449 ━━━━━━━━━━━━━━━━━━━━ 61s 136ms/step - accuracy: 0.5001 - loss: 1.3113 - val_accuracy: 0.5587 - val_loss: 1.1598
Epoch 28/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - accuracy: 0.5073 - loss: 1.2963

449/449 ━━━━━━━━━━━━━━━━━━━━ 63s 141ms/step - accuracy: 0.5058 - loss: 1.3003 - val_accuracy: 0.5642 - val_loss: 1.1528
Epoch 29/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 63s 140ms/step - accuracy: 0.5098 - loss: 1.2932 - val_accuracy: 0.5411 - val_loss: 1.2187
Epoch 30/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.5156 - loss: 1.2848

449/449 ━━━━━━━━━━━━━━━━━━━━ 62s 137ms/step - accuracy: 0.5112 - loss: 1.2914 - val_accuracy: 0.5670 - val_loss: 1.1428
Epoch 31/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 61s 136ms/step - accuracy: 0.5107 - loss: 1.2835 - val_accuracy: 0.5311 - val_loss: 1.2085
Epoch 32/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 61s 136ms/step - accuracy: 0.5126 - loss: 1.2816 - val_accuracy: 0.5536 - val_loss: 1.1666
Epoch 33/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.5157 - loss: 1.2783

449/449 ━━━━━━━━━━━━━━━━━━━━ 61s 137ms/step - accuracy: 0.5130 - loss: 1.2803 - val_accuracy: 0.5737 - val_loss: 1.1318
Epoch 34/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 61s 136ms/step - accuracy: 0.5160 - loss: 1.2774 - val_accuracy: 0.5517 - val_loss: 1.1723
Epoch 35/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 61s 136ms/step - accuracy: 0.5193 - loss: 1.2715 - val_accuracy: 0.5651 - val_loss: 1.1413
Epoch 36/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 61s 136ms/step - accuracy: 0.5180 - loss: 1.2730 - val_accuracy: 0.5662 - val_loss: 1.1696
Epoch 37/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 61s 136ms/step - accuracy: 0.5217 - loss: 1.2552 - val_accuracy: 0.5687 - val_loss: 1.1447
Epoch 38/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 61s 136ms/step - accuracy: 0.5200 - loss: 1.2584 - val_accuracy: 0.5734 - val_loss: 1.1362
Epoch 39/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 62s 137ms/step - accuracy: 0.5220 - loss: 1.2593 - val_accuracy: 0.5670 - val_loss: 1.1373
Epoch 40/60
449/449 ━━━━━━━━━━━━━━━━━━━━ 62s 139ms/step - accuracy: 0.5245 - loss: 1.256

In [15]:
emotion_labels = [
    "Angry", 
    "Disgust", 
    "Fear", 
    "Happy", 
    "Sad", 
    "Surprise", 
    "Neutral"
]

In [2]:
import cv2
import numpy as np
import tensorflow
from tensorflow.keras.models import load_model

# Load trained model
model = load_model("emotion_model.h5")

emotion_labels = [
    "Angry", 
    "Disgust", 
    "Fear", 
    "Happy", 
    "Sad", 
    "Surprise", 
    "Neutral"
]

# Load face detector (Haar cascade)
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    for (x, y, w, h) in faces:
        face = gray[y:y+h, x:x+w]

        # Resize to 48x48 (same as training)
        face = cv2.resize(face, (48, 48))

        # Normalize like training
        face = face / 255.0

        # Reshape to model input
        face = np.reshape(face, (1, 48, 48, 1))

        # Predict
        predictions = model.predict(face, verbose=0)
        emotion = emotion_labels[np.argmax(predictions)]

        # Draw rectangle + label
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
        cv2.putText(frame, emotion, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.9, (0,255,0), 2)

    cv2.imshow("Emotion Detector", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
